# Proyecto Oráculo · Notebook de búsqueda (ADAG1, Grupo 1)

Lenuard · Análisis y Diseño de Algoritmos 2026-B · primera entrega

Este notebook documenta **cómo se obtuvo** la configuración de `entrega.json`:

1. **Diagnóstico por trazas**: cada opción se mide sola y las trazas de fallo (`violo`) dicen qué familia rompe. Se conservan las `K` mejores por ranura, y el espacio pasa de 8⁵ = 32 768 a K⁵ configs.
2. **Búsqueda** en el espacio reducido con los algoritmos del enunciado: backtracking con poda (Alg. 1), Successive Halving (Alg. 2), descenso por coordenadas (Alg. 3) y reparto de presupuesto uniforme frente a adaptativo (Alg. 4). Cada uno tiene su propio contador de consultas con tope `B`.
3. **Selección**: la config de mayor precisión en el lote de búsqueda. Es la **mejor observada**, no un óptimo garantizado.
4. **Validación** con `oraculo.validar` sobre la partición de validación, que no se usó para buscar.
5. **Entrega**: `entrega.json` con esa misma config.

Necesita **GPU T4**: *Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU*.

## 1 · Instalar y clonar el repo · *al terminar, reinicia el entorno de ejecución*

In [ ]:
# bitsandbytes: checkpoints 4-bit. nltk/spacy/emoji/langdetect: los usa el
# verificador de open-instruct. Copia de la celda 1 del notebook del curso, pero
# clonando el fork del grupo en vez de bajar los .py sueltos.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q

%cd /content
# oraculo.py busca los verificadores en /content/open-instruct.
!test -d open-instruct || git clone -q https://github.com/allenai/open-instruct
# El repo del grupo trae oraculo.py, ayudas.py, datos_visibles.json, busqueda/ y entrega.json.
!test -d ADA-2026-2-GP01 && git -C ADA-2026-2-GP01 pull -q || git clone -q https://github.com/Lenuard/ADA-2026-2-GP01

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")

## 2 · Modelo y oráculo

El mismo modelo y los mismos parámetros de lote que el notebook del curso. Si Colab se desconecta, el caché en Drive conserva las generaciones ya pagadas (descomentar las líneas).

In [ ]:
%cd /content/ADA-2026-2-GP01
import sys
sys.path.insert(0, "/content/ADA-2026-2-GP01")

from ayudas import cargar_modelo

MODELO = "qwen17b"
modelo = cargar_modelo(MODELO)

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo
from ayudas import cargar_datos, dividir, curva, entrega

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion, lote=16, presupuesto=32_000_000)
# oraculo = Oraculo(modelo, busqueda, validacion, cache="/content/drive/MyDrive/oraculo_cache.json", lote=16, presupuesto=32_000_000)

## 3 · Parámetros

Cada consulta es un par (config, instancia), es decir **una generación del LLM en la T4**. Por eso el costo se cuenta en consultas.

| Parámetro | Valor | Por qué |
|---|---|---|
| `K` | 2 | Opciones conservadas por ranura. Con 2, \|C'\| = 2⁵ = 32: recorrer todo C' con 100 instancias cuesta 3 200 consultas, frente a 3 276 800 en el espacio completo. |
| `N_DIAG` | 50 | Instancias del diagnóstico. Son (1 + 5·8)·50 = 2 050 consultas; con 50, una opción que rompe una familia frecuente ya se separa de la base. |
| `INST` | las 100 de búsqueda | Lote con el que se comparan las configs. El puntaje es todo-o-nada; con menos instancias muchas configs empatan en 0. |
| `N0` | 3 | Successive Halving: R = log₂32 = 5 rondas con 3, 6, 12, 24 y 48 instancias, y costo nominal 32·3·5 = 480. |
| `B` | 3 200 | Tope de consultas **por algoritmo** (= \|C'\|·100, el costo de la fuerza bruta en C'). Ninguno puede gastar más. |
| `MAX_ITER` | 10 | Tope del descenso por coordenadas, que además para cuando una iteración no cambia nada. |
| `SEMILLA` | 1 | Semilla de generación; en temperatura 0.0 el decode es greedy. |

La temperatura queda fija en **0.0** en esta entrega.

In [ ]:
GRUPO = "G01"
SEMANA = 8
SEMILLA = 1
K = 2
N_DIAG = 50
INST = busqueda          # las 100 instancias de búsqueda
N0 = 3
B = 3200
MAX_ITER = 10
BS_REPARTO = [160, 480, 960, 1600]   # presupuestos para comparar repartos (Alg. 4)

## 4 · (a) Diagnóstico por trazas → espacio reducido

Primero se mide la **config vacía** (sin ninguna ranura, solo la petición) y después cada una de las 40 opciones **sola**. Para cada opción la tabla muestra:

- `precision` y `delta` frente a la base;
- `familias_de_mas`: las familias `violo` que aparecen más veces que en la base, es decir, lo que esa opción rompe;
- `conservada` y `motivo`: la decisión que se tomó.

In [ ]:
import pandas as pd
from collections import Counter

from busqueda import (
    OraculoContado, backtracking, comparar_repartos, descenso_coordenadas,
    diagnosticar, espacio_reducido, ordenar, successive_halving,
    validar_config, validar_entrega,
)

pd.set_option("display.max_colwidth", 120)

INST_DIAG = INST[:N_DIAG]
opciones, tabla = diagnosticar(oraculo, INST_DIAG, k=K, semilla=SEMILLA)
df_diag = pd.DataFrame(tabla)
df_diag["precision"] = df_diag["precision"].map("{:.1%}".format)
df_diag["delta"] = df_diag["delta"].map("{:+.1%}".format)
df_diag[["ranura", "opcion", "precision", "delta", "conservada", "motivo"]]

In [ ]:
base = oraculo.evaluar({"temperatura": 0.0}, INST_DIAG, semilla=SEMILLA)
print(f"config vacía: {base.precision:.1%} en {base.n} instancias")
print("familias que ya fallan sin ninguna ranura:",
      Counter(t["violo"] for t in base.trazas).most_common(5))
print()
for ranura, ops in opciones.items():
    descartadas = [f["opcion"] for f in tabla if f["ranura"] == ranura and not f["conservada"]]
    print(f"{ranura:10}  conservadas {ops}   descartadas {descartadas}")

C_RED = espacio_reducido(opciones)
print(f"\nespacio: 32768 → {len(C_RED)} configs")

**Qué decisiones salen de las trazas** *(completar después de correr; reemplazar con lo observado)*:

- Por cada ranura, qué familias rompen las opciones descartadas, según la columna `motivo`. Por ejemplo, una opción de `cierre` que agrega un "P.S." dispara `detectable_content:postscript` o `startend:end_checker`.
- Esas opciones **no entran** en C'. Ningún algoritmo de la sección (b) las vuelve a evaluar, y el presupuesto se gasta solo en combinar las 2 que quedan por ranura.
- Qué cambia en la búsqueda: pasa de 32 768 configs a 32, y la fuerza bruta en C' queda en 3 200 consultas. El costo es que una opción mala sola pero buena en combinación quedaría fuera. El diagnóstico no ve interacciones, y por eso se conservan 2 por ranura y no 1.

## 5 · (b) Los cuatro algoritmos sobre C'

Cada algoritmo usa su propio `OraculoContado(oraculo, B)`, que cuenta las consultas **nuevas para ese algoritmo**. El caché del oráculo es compartido, así que en la GPU solo se paga la unión, pero la comparación entre algoritmos es justa.

In [ ]:
cont_bt = OraculoContado(oraculo, B, "backtracking")
r_bt = backtracking(cont_bt, opciones, INST, semilla=SEMILLA)
print("Alg. 1 backtracking:", r_bt["config"], f"{r_bt['precision']:.1%}",
      f"| nodos {r_bt['nodos']}, podas {r_bt['podas']}, consultas {cont_bt.consultas}/{B}")

In [ ]:
cont_sh = OraculoContado(oraculo, B, "successive_halving")
r_sh = successive_halving(cont_sh, C_RED, INST, N0, semilla=SEMILLA)
print("Alg. 2 Successive Halving:", r_sh["config"], f"{r_sh['precision']:.1%} (en su última ronda)")
print(f"  rondas {r_sh['rondas']}, consultas nominales {r_sh['consultas_nominales']} "
      f"= fórmula |C|·n0·log2|C| = {r_sh['consultas_formula']:.0f}; reales {cont_sh.consultas}")
pd.DataFrame(r_sh["historia"])

In [ ]:
cont_cd = OraculoContado(oraculo, B, "coordenadas")
r_cd = descenso_coordenadas(cont_cd, opciones, INST, max_iter=MAX_ITER, semilla=SEMILLA)
print("Alg. 3 descenso por coordenadas:", r_cd["config"], f"{r_cd['precision']:.1%}",
      f"| iteraciones {r_cd['iteraciones']}, convergió {r_cd['convergio']}, consultas {cont_cd.consultas}/{B}")

In [ ]:
# Alg. 4: mismo B para los dos repartos. La ganadora de cada uno se mide sobre
# las 100 instancias (sin cobrar) para ver cuál elige mejor.
filas_rep = comparar_repartos(oraculo, C_RED, INST, BS_REPARTO, semilla=SEMILLA)
df_rep = pd.DataFrame(filas_rep)
df_rep

In [ ]:
import matplotlib.pyplot as plt

for metodo, g in df_rep.groupby("metodo"):
    plt.plot(g["B"], g["precision_referencia"], marker="o", label=metodo)
plt.xlabel("presupuesto B (consultas)")
plt.ylabel("precisión de la ganadora (100 inst)")
plt.title("Alg. 4 · reparto uniforme frente a adaptativo")
plt.legend()
plt.show()

### Registro de todas las configs evaluadas

In [ ]:
registro = cont_bt.registro + cont_sh.registro + cont_cd.registro
df_reg = pd.DataFrame(registro)
df_reg["config"] = df_reg["config"].map(lambda c: str(ordenar(c)))
df_reg["completa"] = df_reg["config"].map(lambda s: all(r in s for r in ("rol", "estrategia", "formato", "estilo", "cierre")))
df_reg.to_csv("registro_busqueda.csv", index=False)
print(f"{len(df_reg)} evaluaciones registradas (guardadas en registro_busqueda.csv)")
df_reg.sort_values("precision", ascending=False)

In [ ]:
for cont in (cont_bt, cont_sh, cont_cd):
    print(cont.algoritmo, f"— {cont.consultas} consultas")
    curva(cont.historial)

## 6 · (c) La mejor observada

La ganadora de cada algoritmo se vuelve a medir sobre las **mismas 100 instancias** (Successive Halving reporta su precisión en 48), y se queda la de mayor precisión.

Es la **mejor observada**, no el óptimo garantizado:
1. El diagnóstico descartó opciones por su efecto aislado, sin ver interacciones.
2. Con el LLM, la cota del backtracking es heurística: una ranura puede arreglar fallos de otra, así que la poda puede cortar ramas buenas.
3. Con 100 instancias todo-o-nada, la precisión trae un error estándar de unos ±4 puntos (calculado abajo). Dos configs a menos de eso no se distinguen.

In [ ]:
import math

finales = []
for nombre, r in (("backtracking", r_bt), ("successive_halving", r_sh), ("coordenadas", r_cd)):
    if r["config"] is None:
        continue
    c = ordenar(r["config"])
    p = oraculo.evaluar(c, INST, semilla=SEMILLA).precision
    finales.append({"algoritmo": nombre, "config": c, "precision": p})
df_fin = pd.DataFrame(finales)
display(df_fin)

# max devuelve la primera en empate: el orden de la lista desempata.
ganadora = max(finales, key=lambda f: f["precision"])
mejor = ganadora["config"]
validar_config(mejor)
p_busq = ganadora["precision"]
ee = math.sqrt(p_busq * (1 - p_busq) / len(INST))
print(f"MEJOR OBSERVADA ({ganadora['algoritmo']}): {mejor}")
print(f"precisión en búsqueda: {p_busq:.1%} ± {ee:.1%} (error estándar, n={len(INST)})")

### Trazas de la mejor: qué sigue fallando

In [ ]:
r_mejor = oraculo.evaluar(mejor, INST, semilla=SEMILLA)
print(Counter(t["violo"] for t in r_mejor.trazas).most_common(10))
for t in r_mejor.trazas[:3]:
    print("\nviolo:", t["violo"])
    print(t["salida"][:300])

*(Completar después de correr.)* ¿Las familias que siguen fallando dependen de la configuración (alguna opción elegida todavía las rompe) o son límites del modelo que también fallan con la config vacía? Comparar con las familias de la base de la sección (a).

## 7 · (d) Validación final con `oraculo.validar`

Se mide la config elegida sobre las **87 instancias de validación**, que no se usaron para buscar ni para elegir. Esto comprueba si la config generaliza a instancias nuevas de las mismas familias. **No** es la evaluación final: la profesora califica `entrega.json` con datos privados que pueden traer familias nuevas.

In [ ]:
N_VAL = 87
r_val = oraculo.validar(mejor, n=N_VAL)

ee_val = math.sqrt(r_val.precision * (1 - r_val.precision) / r_val.n)
print(f"\nbúsqueda:   {p_busq:.1%} ± {ee:.1%}")
print(f"validación: {r_val.precision:.1%} ± {ee_val:.1%}")
print(f"diferencia: {r_val.precision - p_busq:+.1%}")

**Interpretación** *(completar con los números obtenidos)*:

- Si la diferencia entre búsqueda y validación cabe dentro de los errores estándar, la config no está sobreajustada al lote de búsqueda.
- Si validación queda claramente por debajo, la selección aprovechó ruido del lote de búsqueda. La precisión reportada en búsqueda es optimista.
- Las familias que imprime `validar` deberían parecerse a las de la sección (c). Una familia nueva ahí indica una superficie que el lote de búsqueda no cubría.

## 8 · (e) Entrega

`ayudas.entrega` escribe `entrega.json` con la **misma** config `mejor` de las secciones (c) y (d). Después hay que subir ese archivo a la raíz del repo (`ADA-2026-2-GP01/entrega.json`) y commitearlo: el notebook de evaluación lo lee de ahí.

In [ ]:
d = entrega(GRUPO, mejor, SEMANA, ruta="entrega.json")
validar_entrega(d)

from google.colab import files
files.download("entrega.json")
files.download("registro_busqueda.csv")